### Loading of the dataset

In [1]:
import os
from datasets import load_dataset

ds = load_dataset("maveriq/tinystoriesv2_gpt4")
cwd = os.getcwd()
ds.save_to_disk(os.path.join(cwd, "tinystories_dataset"))

c:\Users\david_bbnm\OneDrive\Documents\Virtual_Enviorments\ai_dev_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\david_bbnm\OneDrive\Documents\Virtual_Enviorments\ai_dev_venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\david_bbnm\.cache\huggingface\hub\datasets--maveriq--tinystoriesv2_gpt4. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activ

Load The Dataset

In [1]:
import os
from datasets import load_from_disk
from transformers import AutoTokenizer


dataset_path = os.path.join(os.getcwd(), "tinystories_dataset") 


ds = load_from_disk(dataset_path)
print(ds)

c:\Users\david_bbnm\OneDrive\Documents\Virtual_Enviorments\ai_dev_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2717699
    })
    valid: Dataset({
        features: ['text'],
        num_rows: 27630
    })
})


Tokenizer

In [5]:
from transformers import (
    AutoTokenizer, 
    LlamaConfig, 
    LlamaForCausalLM, 
)

# --- 1. TOKENIZER MODERNO ---
# Usiamo il tokenizer di TinyLlama (basato su Llama 2). 
# È molto più pulito di quello di GPT-2.
tokenizer_id = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
tokenizer = AutoTokenizer.from_pretrained(tokenizer_id)
# Llama non ha un pad token di default, lo impostiamo
tokenizer.pad_token = tokenizer.eos_token 

# Preprocessing

In [ ]:
import os
import multiprocessing

# Define global variables
block_size = 512 

# Determine the number of CPUs to use (leave one free)
num_cpu = max(1, os.cpu_count() - 1)

# Function to tokenize the input text
def tokenize_function(examples, tokenizer=None):
    # Use the tokenizer passed as an argument
    return tokenizer(examples["text"])

# Function to group texts into chunks of block_size
def group_texts(examples, block_size=512):
    # 1. Concatenate all texts
    # Create an empty dictionary to hold the flattened lists
    concatenated_examples = {}
    
    # Iterate over every key in the dataset (e.g., 'input_ids', 'attention_mask')
    for key in examples.keys():
        flattened_list = []
        list_of_lists = examples[key]
        
        # Explicit loop to flatten the list of lists
        for sublist in list_of_lists:
            # .extend() adds elements of the sublist to the flat list
            flattened_list.extend(sublist)
            
        concatenated_examples[key] = flattened_list

    # Calculate total length based on the first key (usually input_ids)
    first_key = list(examples.keys())[0]
    total_length = len(concatenated_examples[first_key])
    
    # 2. Truncate to the last complete block
    if total_length >= block_size:
        # Integer division to find exact number of blocks
        total_length = (total_length // block_size) * block_size
        
    # 3. Split into chunks (The actual chunking)
    result = {}
    
    # Loop through each feature (input_ids, attention_mask, etc.)
    for key, values in concatenated_examples.items():
        chunks = []
        # Loop from 0 to total_length with steps of block_size
        for i in range(0, total_length, block_size):
            # Slice the list to get the chunk
            chunk = values[i : i + block_size]
            chunks.append(chunk)
        
        # Assign the list of chunks to the result dictionary
        result[key] = chunks
    
    # 4. Create labels (Clone input_ids)
    # Necessary for Causal Language Modeling
    result["labels"] = result["input_ids"].copy()
    
    return result

# --- Main Execution Pipeline ---

print(f"⏳ Starting Tokenization using {num_cpu} processes...")

# Apply tokenization
tokenized_datasets = ds.map(
    tokenize_function,
    batched=True,
    batch_size=2000,   # Process 2000 samples at a time
    num_proc=num_cpu,  # Use multiple CPU cores
    remove_columns=["text"], # Remove raw text to save RAM
    # Pass arguments explicitly to the function
    fn_kwargs={"tokenizer": tokenizer} 
)

print(f"⏳ Starting Grouping (Chunking)...")

# Apply grouping/chunking
lm_dataset = tokenized_datasets.map(
    group_texts,
    batched=True,
    batch_size=2000,
    num_proc=num_cpu,
    # Pass arguments explicitly to the function
    fn_kwargs={"block_size": block_size}
)

print(f"✅ Dataset processed successfully.")
print(f"   Train size: {len(lm_dataset['train'])}")
print(f"   Valid size: {len(lm_dataset['valid'])}")

⏳ Inizio Tokenizzazione...


Map (num_proc=4):   6%|▌         | 156000/2717699 [01:06<09:06, 4687.75 examples/s]

# Definition of the architecture

In [ ]:
# --- 2. ARCHITETTURA MODERNA () ---
# Calibrazione per ~30 Milioni di parametri
config = LlamaConfig(
    vocab_size=len(tokenizer), # Solitamente 32000
    hidden_size=288,           # Dimensione vettore (più piccolo per compensare SwiGLU)
    intermediate_size=768,     # SwiGLU richiede questo ~2.5x hidden_size
    num_hidden_layers=8,       # Più profondo è meglio
    num_attention_heads=6,     # 288 / 6 = 48 dim per head
    num_key_value_heads=6,     # Grouped Query Attention (opzionale, qui teniamo uguale)
    hidden_act="silu",         # SiLU è la componente chiave di SwiGLU
    max_position_embeddings=512, # RoPE supporta contesti più lunghi nativamente
    rms_norm_eps=1e-5,
    rope_theta=10000.0,        # Parametro per la rotazione
)

# Inizializzazione modello
model = LlamaForCausalLM(config) # attn_implementation="flash_attention_2 se avessi colab

# --- 3. VERIFICA PARAMETRI ---
num_params = sum(p.numel() for p in model.parameters())
print(f"🦙 Inizializzato TinyLlama Custom!")
print(f"📊 Parametri Totali: {num_params / 1_000_000:.2f}M")


Set up of the training

In [ ]:
import os
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

cwd = os.getcwd()
# --- 2. CONFIGURAZIONE TRAINER ---

# Data Collator per Causal LM
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# Argomenti di Training
training_args = TrainingArguments(
    output_dir=os.path.join(cwd, "model_checkpoints"),
    overwrite_output_dir=True,
    num_train_epochs=7, 
    
    # Batch Size
    per_device_train_batch_size=32, 
    gradient_accumulation_steps=4,  
    
    # Precisione
    fp16=True,
    
    # Ottimizzazione
    learning_rate=5e-4,             
    warmup_steps=500,               
    weight_decay=0.01,
    
    # --- NUOVO: Configurazione VALIDAZIONE ---
    eval_strategy="steps",      # Valuta ogni X step (non solo a fine epoca)
    eval_steps=2000,            # Valuta ogni 2000 step (come il save)
    save_steps=2000,            # Salva ogni 2000 step
    logging_steps=100,
    
    save_total_limit=1, 
    load_best_model_at_end=True, 
    metric_for_best_model="eval_loss",
    greater_is_better=False,    
)

# Inizializzazione Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    
    # Passiamo i dataset processati
    train_dataset=lm_dataset["train"], 
    eval_dataset=lm_dataset["valid"],   
    data_collator=data_collator,
    tokenizer=tokenizer,
)

# AVVIO
print("🚀 Inizio Training...")
trainer.train()

# Salvataggio Finale
trainer.save_model("tiny_stories_final")
print("🏁 Training completato e modello salvato!")

# Alla fine dello script, dopo trainer.save_model()
tokenizer.save_pretrained("tiny_stories_final")

### Testing

In [ ]:
import torch
from transformers import LlamaForCausalLM, AutoTokenizer

# --- CONFIGURAZIONE ---
model_path = "tiny_stories_final"  # La cartella dove hai salvato il modello
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"⚙️ Utilizzo device: {device}")

# 1. Caricamento (se non sono già in memoria)
try:
    model = LlamaForCausalLM.from_pretrained(model_path).to(device)
    tokenizer = AutoTokenizer.from_pretrained(model_path)
except Exception as e:
    print("Model Not Found!!")

# 2. Funzione di Generazione
def generate_text(prompt, max_length=200):
    model.eval() # Modalità inferenza
    
    # Tokenizzazione
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    # Generazione
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_length,
            do_sample=True,         # Attiva la creatività (sampling)
            temperature=0.7,        # 0.7 è un buon bilanciamento tra creatività e coerenza
            top_k=50,               # Prende le top 50 parole probabili
            top_p=0.95,             # Nucleus sampling
            repetition_penalty=1.2, # Penalizza le ripetizioni (fondamentale per modelli piccoli)
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decodifica
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# --- TEST ---
prompt = "Once on an island"

model_generation = generate_text(prompt)
print("Out text:\n\n", model_generation)